# Establishing the Existence of an Assistant Basin

**Research question:** Does the transformer's own layer-by-layer dynamics actively restore the assistant representation when perturbed? If so, the assistant isn't just a geometric feature (an axis) — it's a **dynamical attractor** with self-correcting properties.

**Method:** Perturb the residual stream at various layers, in various directions and magnitudes. Measure whether subsequent layers pull the representation back toward the baseline (recovery) or let it diverge. Compare recovery rates for:
- **Assistant axis direction** (away from assistant)
- **Assistant axis direction** (toward assistant / overshoot)
- **Random directions** (control)

If the assistant direction recovers faster than random → specific restoring force exists.
If away and toward both recover symmetrically → true basin (not just directional bias).

**Built on:** [The Assistant Axis](https://arxiv.org/abs/2601.10387) (Lu et al., 2026)

## 1. Setup

In [ ]:
# Setup:
#   git clone https://github.com/maxxts7/assitant_basin.git && cd assitant_basin
#   pip install -r requirements.txt
#   jupyter notebook --allow-root
#
# No need to clone assistant-axis — basin_experiment.py is self-contained
# and downloads pre-computed axis vectors from HuggingFace automatically.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from basin_experiment import BasinExperiment, DEFAULT_PROMPTS
from basin_analysis import (
    plot_recovery_curves,
    plot_asymmetry,
    plot_basin_heatmap,
    plot_basin_width,
    test_directional_recovery,
    test_symmetry,
    print_summary,
    generate_all_plots,
)

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name} ({mem:.1f} GB)")

In [ ]:
# ---- Configuration ----
# Change MODEL_NAME to match your hardware:
#   - "google/gemma-2-27b-it"                → ~54GB, 1x A100-80GB or 2x A100-40GB
#   - "Qwen/Qwen3-32B"                       → ~64GB, 2x A100-40GB
#   - "meta-llama/Llama-3.3-70B-Instruct"    → ~140GB, 2x A100-80GB

MODEL_NAME = "Qwen/Qwen3-32B"

# Experiment parameters
PERTURB_LAYERS = [0, 7, 14, 21, 28, 35, 42, 49, 56, 63]  # Qwen3-32B has 64 layers
ALPHAS = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0, 2.0]
N_RANDOM_DIRS = 5
SEED = 42

In [ ]:
# Load model and axis (deterministic=True disables flash attention and forces deterministic CUDA ops)
exp = BasinExperiment(MODEL_NAME, deterministic=True)

print(f"Model: {MODEL_NAME}")
print(f"Layers: {exp.num_layers}")
print(f"Hidden dim: {exp.hidden_dim}")
print(f"Axis shape: {exp.axis.shape}")
print(f"Perturbation layers: {PERTURB_LAYERS}")
print(f"Alpha values: {ALPHAS}")
print(f"Directions per layer: 2 (assistant) + {N_RANDOM_DIRS} (random) = {2 + N_RANDOM_DIRS}")

## 2. Sanity Check

Before running the full experiment, verify on a single prompt:
1. Baseline activations are deterministic (same result on two runs)
2. Perturbation actually changes downstream activations
3. Recovery metrics look reasonable

In [ ]:
# Sanity check 1: Determinism
test_prompt = "What is the capital of France?"
input_ids = exp.tokenize(test_prompt)

baseline_1, logits_1 = exp.get_baseline_trajectory(input_ids)
baseline_2, logits_2 = exp.get_baseline_trajectory(input_ids)

max_diff = max(
    (baseline_1[l] - baseline_2[l]).abs().max().item()
    for l in baseline_1
)
print(f"Max activation difference between two baseline runs: {max_diff:.2e}")

if max_diff < 1e-5:
    print("Baseline is fully deterministic.")
elif max_diff < 1e-1:
    print("Small non-determinism detected (expected with bf16 / multi-GPU / flash attention).")
    print("This is fine — perturbation magnitudes will be orders of magnitude larger.")
else:
    print("WARNING: Large non-determinism detected. Results may be noisy.")
    print("Consider setting: torch.backends.cudnn.deterministic = True")

In [ ]:
# Sanity check 2: Build all 4 directions + run 7 perturbations
from basin_experiment import make_perturbation, MODEL_CONFIGS

test_layer = MODEL_CONFIGS.get(MODEL_NAME, {}).get("target_layer", exp.num_layers // 2)
h_L = baseline_1[test_layer]

# ---- Direction 1: Assistant axis ----
axis_dir = exp.axis[test_layer].float()
axis_dir = axis_dir / axis_dir.norm()

# ---- Direction 2: Random (low-variance, no meaning) ----
torch.manual_seed(0)
random_dir = torch.randn(exp.hidden_dim)
random_dir = random_dir / random_dir.norm()

# ---- Direction 3: Factual vs Creative contrast (meaningful, high-variance) ----
factual_prompts = [
    "What is the boiling point of water?",
    "How many bones are in the human body?",
    "What is the chemical formula for table salt?",
    "When did World War II end?",
    "What is the speed of sound in air?",
    "How far is the moon from Earth?",
    "What is the atomic number of carbon?",
    "Who invented the telephone?",
    "What is the largest planet in our solar system?",
    "How many chromosomes do humans have?",
    "What is the freezing point of mercury?",
    "What year was the internet invented?",
    "How long does light take to reach Earth from the Sun?",
    "What is the population of Tokyo?",
    "What is the pH of pure water?",
]
creative_prompts = [
    "Write a poem about a dying star.",
    "Describe a color that doesn't exist.",
    "Write the opening of a novel set in a dream.",
    "Invent a new mythological creature and describe it.",
    "Write a love letter from the ocean to the moon.",
    "Describe what silence sounds like.",
    "Write a short story about a clock that runs backwards.",
    "Imagine a conversation between fire and ice.",
    "Describe the taste of a memory.",
    "Write a monologue from the perspective of a black hole.",
    "Create a recipe for happiness.",
    "Write a eulogy for the last tree on Earth.",
    "Describe a sunset to someone who has never seen light.",
    "Write a fairy tale set in a quantum computer.",
    "Invent a new emotion and describe how it feels.",
]

print(f"Computing factual vs creative contrast from {len(factual_prompts)}+{len(creative_prompts)} prompts...")
factual_acts = []
for p in factual_prompts:
    ids = exp.tokenize(p)
    acts, _ = exp.get_baseline_trajectory(ids)
    factual_acts.append(acts[test_layer])

creative_acts = []
for p in creative_prompts:
    ids = exp.tokenize(p)
    acts, _ = exp.get_baseline_trajectory(ids)
    creative_acts.append(acts[test_layer])

factual_mean = torch.stack(factual_acts).mean(dim=0).float()
creative_mean = torch.stack(creative_acts).mean(dim=0).float()
fc_dir = factual_mean - creative_mean
fc_dir_raw_norm = fc_dir.norm().item()
fc_dir = fc_dir / fc_dir.norm()

# Orthogonalize against assistant axis
fc_cos_with_axis = (fc_dir @ axis_dir).item()
fc_dir = fc_dir - (fc_dir @ axis_dir) * axis_dir
fc_dir = fc_dir / fc_dir.norm()

# ---- Direction 4: PCA PC1 from diverse prompts (high-variance, no meaning) ----
pca_prompts = factual_prompts + creative_prompts  # reuse the 30 prompts we already ran
all_pca_acts = factual_acts + creative_acts

print(f"Computing PCA from {len(all_pca_acts)} prompt activations...")
act_matrix = torch.stack(all_pca_acts).float()
act_centered = act_matrix - act_matrix.mean(dim=0)
U, S, Vt = torch.linalg.svd(act_centered, full_matrices=False)

pc1_dir = Vt[0]
pc1_raw_cos_with_axis = (pc1_dir @ axis_dir).item()
pc1_raw_cos_with_fc = (pc1_dir @ fc_dir).item()

total_var = (S ** 2).sum().item()
pc1_var = (S[0] ** 2).item() / total_var * 100
pc2_var = (S[1] ** 2).item() / total_var * 100
pc3_var = (S[2] ** 2).item() / total_var * 100

# Orthogonalize PC1 against both assistant axis AND factual-creative
pc1_dir = pc1_dir - (pc1_dir @ axis_dir) * axis_dir
pc1_dir = pc1_dir - (pc1_dir @ fc_dir) * fc_dir
pc1_dir = pc1_dir / pc1_dir.norm()

# ---- Print direction diagnostics ----
print(f"\n{'='*60}")
print(f"DIRECTION DIAGNOSTICS (layer {test_layer})")
print(f"{'='*60}")
print(f"Baseline activation norm: {h_L.norm().item():.2f}")
print(f"Baseline projection onto assistant axis: {(h_L @ axis_dir).item():.4f}")
print()
print(f"Factual-Creative contrast:")
print(f"  Raw norm (before normalization): {fc_dir_raw_norm:.2f}")
print(f"  cos(FC, assistant_axis) before ortho: {fc_cos_with_axis:.4f}")
print(f"  cos(FC, assistant_axis) after ortho:  {(fc_dir @ axis_dir).item():.6f}")
print()
print(f"PCA (from {len(all_pca_acts)} prompts):")
print(f"  Variance explained: PC1={pc1_var:.1f}%  PC2={pc2_var:.1f}%  PC3={pc3_var:.1f}%")
print(f"  cos(PC1, assistant_axis) before ortho: {pc1_raw_cos_with_axis:.4f}")
print(f"  cos(PC1, FC_dir) before ortho:         {pc1_raw_cos_with_fc:.4f}")
print(f"  cos(PC1, assistant_axis) after ortho:  {(pc1_dir @ axis_dir).item():.6f}")
print(f"  cos(PC1, FC_dir) after ortho:          {(pc1_dir @ fc_dir).item():.6f}")
print()
print(f"Cross-direction cosines (all after ortho):")
print(f"  cos(assistant, random):  {(axis_dir @ random_dir).item():.4f}")
print(f"  cos(assistant, FC):      {(axis_dir @ fc_dir).item():.6f}")
print(f"  cos(assistant, PC1):     {(axis_dir @ pc1_dir).item():.6f}")
print(f"  cos(FC, random):         {(fc_dir @ random_dir).item():.4f}")
print(f"  cos(FC, PC1):            {(fc_dir @ pc1_dir).item():.6f}")
print(f"  cos(PC1, random):        {(pc1_dir @ random_dir).item():.4f}")

# ---- Run all 7 perturbations ----
alpha = 0.5
print(f"\n{'='*60}")
print(f"PERTURBATIONS (alpha={alpha}, magnitude={make_perturbation(h_L, axis_dir, alpha).norm().item():.2f})")
print(f"{'='*60}")

delta_away    = make_perturbation(h_L, -axis_dir, alpha)
delta_toward  = make_perturbation(h_L,  axis_dir, alpha)
delta_random  = make_perturbation(h_L,  random_dir, alpha)
delta_fc_pos  = make_perturbation(h_L,  fc_dir, alpha)    # toward factual
delta_fc_neg  = make_perturbation(h_L, -fc_dir, alpha)    # toward creative
delta_pc1_pos = make_perturbation(h_L,  pc1_dir, alpha)
delta_pc1_neg = make_perturbation(h_L, -pc1_dir, alpha)

away_acts, away_logits       = exp.get_perturbed_trajectory(input_ids, test_layer, delta_away)
toward_acts, toward_logits   = exp.get_perturbed_trajectory(input_ids, test_layer, delta_toward)
random_acts, random_logits   = exp.get_perturbed_trajectory(input_ids, test_layer, delta_random)
fc_pos_acts, fc_pos_logits   = exp.get_perturbed_trajectory(input_ids, test_layer, delta_fc_pos)
fc_neg_acts, fc_neg_logits   = exp.get_perturbed_trajectory(input_ids, test_layer, delta_fc_neg)
pc1_pos_acts, pc1_pos_logits = exp.get_perturbed_trajectory(input_ids, test_layer, delta_pc1_pos)
pc1_neg_acts, pc1_neg_logits = exp.get_perturbed_trajectory(input_ids, test_layer, delta_pc1_neg)

# Compute metrics
away_metrics    = exp.compute_recovery_metrics(baseline_1, away_acts, logits_1, away_logits, test_layer)
toward_metrics  = exp.compute_recovery_metrics(baseline_1, toward_acts, logits_1, toward_logits, test_layer)
random_metrics  = exp.compute_recovery_metrics(baseline_1, random_acts, logits_1, random_logits, test_layer)
fc_pos_metrics  = exp.compute_recovery_metrics(baseline_1, fc_pos_acts, logits_1, fc_pos_logits, test_layer)
fc_neg_metrics  = exp.compute_recovery_metrics(baseline_1, fc_neg_acts, logits_1, fc_neg_logits, test_layer)
pc1_pos_metrics = exp.compute_recovery_metrics(baseline_1, pc1_pos_acts, logits_1, pc1_pos_logits, test_layer)
pc1_neg_metrics = exp.compute_recovery_metrics(baseline_1, pc1_neg_acts, logits_1, pc1_neg_logits, test_layer)

print("Done. Run next cell for full data.")

In [ ]:
# Sanity check 3: Full raw data — all 7 perturbations, all downstream layers

all_dirs = {
    "Away":    away_metrics,     # assistant axis, away
    "Toward":  toward_metrics,   # assistant axis, toward
    "Random":  random_metrics,   # random direction
    "FC+":     fc_pos_metrics,   # factual-creative, toward factual
    "FC-":     fc_neg_metrics,   # factual-creative, toward creative
    "PC1+":    pc1_pos_metrics,  # PCA PC1 positive
    "PC1-":    pc1_neg_metrics,  # PCA PC1 negative
}

print(f"Perturbation at layer {test_layer}, alpha={alpha}")
print(f"Labels: Away/Toward=assistant axis, FC+/FC-=factual-creative, PC1+/PC1-=PCA, Random=noise")

# --- Table 1: Normalized Distance ---
print(f"\n--- Normalized Distance (lower = more recovery) ---")
header = f"{'Layer':<6}" + "".join(f"{k:>8}" for k in all_dirs.keys())
print(header)
print("-" * len(header))
for i in range(len(away_metrics)):
    l = away_metrics[i]["downstream_layer"]
    vals = [m[i]["normalized_distance"] for m in all_dirs.values()]
    print(f"{l:<6}" + "".join(f"{v:>8.4f}" for v in vals))

# --- Table 2: Axis Projection Gap ---
print(f"\n--- Axis Projection Gap (gap along assistant axis) ---")
header = f"{'Layer':<6}" + "".join(f"{k:>8}" for k in all_dirs.keys())
print(header)
print("-" * len(header))
for i in range(len(away_metrics)):
    l = away_metrics[i]["downstream_layer"]
    vals = [m[i]["axis_projection_gap"] for m in all_dirs.values()]
    print(f"{l:<6}" + "".join(f"{v:>8.1f}" for v in vals))

# --- Table 3: Cosine Similarity with baseline ---
print(f"\n--- Cosine Similarity with Baseline (higher = more recovery) ---")
header = f"{'Layer':<6}" + "".join(f"{k:>8}" for k in all_dirs.keys())
print(header)
print("-" * len(header))
for i in range(len(away_metrics)):
    l = away_metrics[i]["downstream_layer"]
    vals = [m[i]["cosine_similarity"] for m in all_dirs.values()]
    print(f"{l:<6}" + "".join(f"{v:>8.4f}" for v in vals))

# --- Summary ---
print(f"\n{'='*70}")
print(f"SUMMARY")
print(f"{'='*70}")
print(f"{'':>20}" + "".join(f"{k:>8}" for k in all_dirs.keys()))
print("-" * 76)
print(f"{'Initial distance':<20}" + "".join(f"{m[0]['normalized_distance']:>8.4f}" for m in all_dirs.values()))
print(f"{'Final distance':<20}" + "".join(f"{m[-1]['normalized_distance']:>8.4f}" for m in all_dirs.values()))
print(f"{'Distance recovery%':<20}" + "".join(f"{(1 - m[-1]['normalized_distance']/m[0]['normalized_distance'])*100:>7.1f}%" for m in all_dirs.values()))
print()
print(f"{'Initial cos sim':<20}" + "".join(f"{m[0]['cosine_similarity']:>8.4f}" for m in all_dirs.values()))
print(f"{'Final cos sim':<20}" + "".join(f"{m[-1]['cosine_similarity']:>8.4f}" for m in all_dirs.values()))
print()
print(f"{'Initial axis gap':<20}" + "".join(f"{m[0]['axis_projection_gap']:>8.1f}" for m in all_dirs.values()))
print(f"{'Final axis gap':<20}" + "".join(f"{m[-1]['axis_projection_gap']:>8.1f}" for m in all_dirs.values()))
print(f"{'Axis gap recovery%':<20}", end="")
for m in all_dirs.values():
    ig = abs(m[0]["axis_projection_gap"])
    fg = abs(m[-1]["axis_projection_gap"])
    if ig > 0.1:
        print(f"{(1 - fg/ig)*100:>7.1f}%", end="")
    else:
        print(f"{'n/a':>8}", end="")
print()
print()
print(f"{'KL divergence':<20}" + "".join(f"{m[0]['kl_divergence']:>8.4f}" for m in all_dirs.values()))
print(f"{'Top-1 preserved':<20}" + "".join(f"{str(bool(m[0]['top1_preserved'])):>8}" for m in all_dirs.values()))

# --- The critical comparison ---
print(f"\n{'='*70}")
print(f"ASYMMETRY CHECK: Does +/- direction matter?")
print(f"{'='*70}")
pairs = [
    ("Assistant", "Away", "Toward", away_metrics, toward_metrics),
    ("Fact/Creat", "FC+", "FC-", fc_pos_metrics, fc_neg_metrics),
    ("PCA PC1", "PC1+", "PC1-", pc1_pos_metrics, pc1_neg_metrics),
]
print(f"{'Direction':<12} {'+ final dist':>12} {'- final dist':>12} {'diff':>8} {'+ final cos':>12} {'- final cos':>12} {'diff':>8}")
print("-" * 82)
for name, pname, nname, pos_m, neg_m in pairs:
    pd_val = pos_m[-1]["normalized_distance"]
    nd_val = neg_m[-1]["normalized_distance"]
    pc_val = pos_m[-1]["cosine_similarity"]
    nc_val = neg_m[-1]["cosine_similarity"]
    print(f"{name:<12} {pd_val:>12.4f} {nd_val:>12.4f} {pd_val-nd_val:>8.4f} {pc_val:>12.4f} {nc_val:>12.4f} {pc_val-nc_val:>8.4f}")

print(f"\nIf Assistant shows large asymmetry but Fact/Creat and PCA don't → assistant-specific effect")
print(f"If all three show similar asymmetry → generic high-variance direction property")

## 3. Full Experiment

Run the complete perturbation-recovery protocol across all prompts, layers, directions, and magnitudes. This is the main compute-intensive step (~4-5 hours on A100-80GB with Gemma 27B).

**What's being computed:**
- 50 prompts × 10 perturbation layers × 7 directions × 7 magnitudes = ~24,500 perturbed forward passes
- Plus 50 baseline forward passes
- At each, we measure recovery at all downstream layers

In [ ]:
# Use first 50 prompts (increase to 100 for full run)
prompts = DEFAULT_PROMPTS[:50]

print(f"Running experiment:")
print(f"  Prompts: {len(prompts)}")
print(f"  Perturbation layers: {PERTURB_LAYERS}")
print(f"  Alphas: {ALPHAS}")
print(f"  Directions: 2 (assistant) + {N_RANDOM_DIRS} (random)")
n_passes = len(prompts) * len(PERTURB_LAYERS) * (2 + N_RANDOM_DIRS) * len(ALPHAS) + len(prompts)
print(f"  Total forward passes: ~{n_passes:,}")

df = exp.run_experiment(
    prompts=prompts,
    perturb_layers=PERTURB_LAYERS,
    alphas=ALPHAS,
    n_random_dirs=N_RANDOM_DIRS,
    seed=SEED,
)

print(f"\nResults shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Save results
df.to_parquet(RESULTS_DIR / "basin_results.parquet", index=False)
print(f"Results saved to {RESULTS_DIR / 'basin_results.parquet'}")

# To reload later:
# df = pd.read_parquet(RESULTS_DIR / "basin_results.parquet")

## 4. Analysis & Visualization

### 4.1 Recovery Curves (the money plot)

The key question: does the assistant axis perturbation recover faster than random?
- **Red** = pushed away from assistant
- **Green** = pushed toward assistant (overshoot)
- **Gray** = random direction (control)

If red/green lines fall below gray → the assistant direction has a specific restoring force.

In [ ]:
# Recovery curves at different magnitudes (averaged over all perturbation layers)
for alpha in [0.1, 0.5, 1.0]:
    fig, ax = plot_recovery_curves(df, alpha=alpha)
    plt.show()

In [ ]:
# Recovery curves at specific perturbation layers (alpha=0.5)
# Compare early, mid, and late layers
key_layers = [
    PERTURB_LAYERS[0],                        # early
    PERTURB_LAYERS[len(PERTURB_LAYERS) // 2],  # mid
    PERTURB_LAYERS[-2],                        # late (not last — need downstream room)
]

for layer in key_layers:
    fig, ax = plot_recovery_curves(df, alpha=0.5, perturb_layer=layer)
    plt.show()

### 4.2 Asymmetry Test

Is the recovery symmetric? If both "away from assistant" and "toward assistant" converge to the same point (gap → 0), that's a **true basin**. If only one direction recovers, it's a directional bias.

In [ ]:
# Asymmetry plots
for alpha in [0.1, 0.5, 1.0]:
    fig, ax = plot_asymmetry(df, alpha=alpha)
    plt.show()

### 4.3 Basin Heatmap

Where is the basin deepest? Green = recovery (inside basin), Red = divergence (outside).

Compare the assistant axis heatmap to the random direction heatmap. If the assistant map is greener, there's a direction-specific basin.

In [ ]:
# Basin heatmaps
fig1, ax1 = plot_basin_heatmap(df, "assistant_away")
plt.show()

fig2, ax2 = plot_basin_heatmap(df, "random")
plt.show()

### 4.4 Basin Width Across Layers

The critical alpha (largest perturbation that still recovers) at each layer.
If assistant_away has a wider basin radius than random at mid-layers, the network specifically tolerates perturbation along the assistant direction.

In [ ]:
# Basin width comparison
fig, ax = plot_basin_width(df)
plt.show()

## 5. Statistical Tests

Two key tests:
1. **Directional recovery test:** Does the assistant axis direction recover significantly faster than random? (paired t-test)
2. **Symmetry test:** Do away-from and toward-assistant perturbations converge symmetrically? (paired t-test on final gap magnitudes)

In [ ]:
# Run statistical tests at multiple alpha values
for alpha in [0.1, 0.5, 1.0]:
    print(f"\n{'='*50}")
    print(f"Alpha = {alpha}")
    print(f"{'='*50}")
    
    directional = test_directional_recovery(df, alpha)
    print(f"\nDirectional Recovery Test:")
    print(f"  Away recovery score:   {directional['mean_away']:.4f}")
    print(f"  Random recovery score: {directional['mean_random']:.4f}")
    print(f"  t={directional['t_statistic']:.3f}, p={directional['p_value']:.2e}, d={directional['cohens_d']:.3f}")
    print(f"  Significant: {directional['significant_at_0.05']}")
    
    symmetry = test_symmetry(df, alpha)
    print(f"\nSymmetry Test:")
    print(f"  Final gap (away):   {symmetry['mean_gap_away']:.4f}")
    print(f"  Final gap (toward): {symmetry['mean_gap_toward']:.4f}")
    print(f"  t={symmetry['t_statistic']:.3f}, p={symmetry['p_value']:.2e}")
    print(f"  Symmetric: {symmetry['symmetric_at_0.05']}")

## 6. Summary & Interpretation

In [ ]:
# Print full summary with conclusion
print_summary(df, alpha=0.5)

In [ ]:
# Save all plots to results/
generate_all_plots(df, output_dir="results")
print("Done. Check results/ for all saved plots.")

## 7. Interpretation Guide

### Reading the results:

| What you see | What it means |
|---|---|
| Red line below gray in recovery curves | Assistant direction recovers faster than random → **restoring force exists** |
| Red and green converge to same level | Symmetric recovery → **true basin** (not just directional bias) |
| Green basin heatmap at mid-layers, red at edges | Basin is deepest in mid-layers (consistent with Fernando & Guitchounts 2025) |
| Assistant basin radius > random basin radius | The network specifically tolerates perturbation along the assistant direction |

### What the conclusions mean:

- **"Evidence supports the existence of an assistant basin"** → The transformer actively self-corrects toward assistant-like representations. This is a stronger claim than the axis paper, which only showed a geometric feature. Safety implication: the model has built-in resistance to persona drift.

- **"Directional restoring force exists, but asymmetric"** → There's a restoring force, but it only works in one direction (e.g., recovering from being pushed away, but not from overshoot). This is a directional bias, not a true attractor. Safety implication: the model resists being pushed away from assistant mode, but overshooting (being "too helpful") is stable.

- **"No evidence for an assistant-specific basin"** → Recovery (if any) is a generic property of the architecture (LayerNorm, etc.), not specific to assistant-ness. The assistant axis is just a geometric feature with no dynamical significance. Safety implication: the axis can be used for measurement but provides no inherent stability.

### Next steps:
1. If positive: Repeat on Qwen 3 32B and Llama 3.3 70B (pre-computed axes available)
2. Map basin shape in more directions (refusal vector, truth direction, PCA components)
3. Compare base vs instruct models: does RLHF deepen the basin?
4. Connect to tuned lens: does the prediction trajectory recover incrementally or chaotically?